# Wine Quality: From Chaos to Order

Workshop opener for *Version Everything: From Chaos to Order in Reproducible Python Projects* — PyCon Italia 2026.

We run the same K-means algorithm on the same wine dataset twice:
- **Round 1** — raw features, no preprocessing → poor separation
- **Round 2** — z-scored + PCA-reduced → clean separation

Same algorithm. Same data. Very different result.  
The difference is the preprocessing pipeline — and whether we can reproduce it.

In [ ]:
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.vq import kmeans2
from scipy.stats import zscore

RANDOM_SEED = 42
N_CLUSTERS = 2
N_COMPONENTS = 2
DATA_DIR = Path('../data')

RED_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'
WHITE_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv'

FEATURES = [
    'fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
    'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
    'pH', 'sulphates', 'alcohol',
]

: 

In [ ]:
def download_if_missing(url: str, dest: Path) -> None:
    if not dest.exists():
        dest.parent.mkdir(parents=True, exist_ok=True)
        print(f'Downloading {dest.name}...')
        urllib.request.urlretrieve(url, dest)
        print(f'  Saved to {dest}')
    else:
        print(f'  {dest.name} already cached.')


def download_data() -> None:
    download_if_missing(RED_URL, DATA_DIR / 'winequality-red.csv')
    download_if_missing(WHITE_URL, DATA_DIR / 'winequality-white.csv')


def load_wine_data() -> pd.DataFrame:
    red = pd.read_csv(DATA_DIR / 'winequality-red.csv', sep=';')
    red['type'] = 'red'
    white = pd.read_csv(DATA_DIR / 'winequality-white.csv', sep=';')
    white['type'] = 'white'
    return pd.concat([red, white], ignore_index=True)


def cluster_accuracy(true_labels: np.ndarray, cluster_labels: np.ndarray) -> float:
    mapping = {}
    for c in np.unique(cluster_labels):
        mask = cluster_labels == c
        mapping[c] = np.bincount(true_labels[mask]).argmax()
    predicted = np.array([mapping[c] for c in cluster_labels])
    return (predicted == true_labels).mean()


def run_kmeans(data: np.ndarray, k: int = 2, seed: int = RANDOM_SEED) -> np.ndarray:
    _, labels = kmeans2(data, k, iter=100, minit='points', seed=seed)
    return labels


def pca_via_svd(data: np.ndarray, n_components: int = 2):
    scaled = zscore(data, axis=0)
    _, S, Vt = np.linalg.svd(scaled, full_matrices=False)
    explained = (S ** 2) / (S ** 2).sum()
    projected = scaled @ Vt[:n_components].T
    return projected, explained

## 1. Download Data

Fetch the UCI Wine Quality CSVs into `data/`. Skips files that are already present.

In [ ]:
download_data()

## 2. Load & Combine

Read the cached CSVs, add a `type` column, and concatenate into a single dataframe.

In [ ]:
df = load_wine_data()
red_n = (df['type'] == 'red').sum()
white_n = (df['type'] == 'white').sum()
print(f'Total rows: {len(df):,}  |  Red: {red_n:,}  |  White: {white_n:,}')
df.head()

## 3. Exploratory Data Analysis

Feature distributions by wine type and quality score distribution.

In [ ]:
cols_to_plot = FEATURES + ['quality']
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, feat in enumerate(cols_to_plot):
    ax = axes[i]
    for wine_type, color in [('red', '#c0392b'), ('white', '#f39c12')]:
        ax.hist(df.loc[df['type'] == wine_type, feat], bins=30, alpha=0.6, color=color, label=wine_type)
    ax.set_title(feat, fontsize=9)
    ax.tick_params(labelsize=7)
    if i == 0:
        ax.legend(fontsize=7)

plt.suptitle('Feature Distributions by Wine Type', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Naive Baseline: K-Means on Raw Features

Can K-means with `k=2` separate red from white wine using the 11 raw features?

Each cluster is mapped to a wine type by majority vote. Accuracy measures how well the clusters recover the true labels.

In [ ]:
X = df[FEATURES].to_numpy()
y = (df['type'] == 'white').astype(int).to_numpy()  # 0 = red, 1 = white

raw_labels = run_kmeans(X, k=N_CLUSTERS)
raw_acc = cluster_accuracy(y, raw_labels)
print(f'Naive accuracy (raw features): {raw_acc:.1%}')

## 5. Diagnose: Why Did It Fail?

K-means relies on Euclidean distance. Features on large scales dominate: a 100-unit difference in *total sulfur dioxide* completely swamps a 0.1-unit difference in *pH*.

The variance bar chart makes the scale problem visible at a glance.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

variances = df[FEATURES].var()
ax1.bar(range(len(FEATURES)), variances.values, color='#3498db')
ax1.set_xticks(range(len(FEATURES)))
ax1.set_xticklabels(FEATURES, rotation=45, ha='right', fontsize=8)
ax1.set_title('Per-Feature Variance (raw)')
ax1.set_ylabel('Variance')

corr = df[FEATURES].corr().to_numpy()
im = ax2.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax2.set_xticks(range(len(FEATURES)))
ax2.set_yticks(range(len(FEATURES)))
ax2.set_xticklabels(FEATURES, rotation=45, ha='right', fontsize=7)
ax2.set_yticklabels(FEATURES, fontsize=7)
ax2.set_title('Feature Correlation Heatmap')
plt.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

## 6. Improved Pipeline: Z-Score + PCA

Two changes:
1. **Z-score normalisation** — each feature gets mean 0, std 1, eliminating scale dominance.
2. **PCA via SVD** — reduce 11 dimensions to 2 for visualisation while preserving most variance.

The scree plot shows how much variance each component captures; the scatter is the visual payoff.

In [ ]:
projected, explained = pca_via_svd(X, n_components=N_COMPONENTS)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

n = len(explained)
cumulative = np.cumsum(explained)
ax1.bar(range(1, n + 1), explained * 100, color='#3498db', label='Individual')
ax1.plot(range(1, n + 1), cumulative * 100, 'o-', color='#e74c3c', label='Cumulative')
ax1.axhline(cumulative[N_COMPONENTS - 1] * 100, color='gray', linestyle='--', alpha=0.5,
            label=f'{N_COMPONENTS} PCs = {cumulative[N_COMPONENTS - 1]:.0%}')
ax1.set_xlabel('Principal Component')
ax1.set_ylabel('Explained Variance (%)')
ax1.set_title('Scree Plot')
ax1.set_xticks(range(1, n + 1))
ax1.legend(fontsize=8)

for wine_type, color in [('red', '#c0392b'), ('white', '#f39c12')]:
    mask = (df['type'] == wine_type).to_numpy()
    ax2.scatter(projected[mask, 0], projected[mask, 1],
                c=color, alpha=0.3, s=10, label=wine_type)
ax2.set_xlabel(f'PC1 ({explained[0]:.1%} variance)')
ax2.set_ylabel(f'PC2 ({explained[1]:.1%} variance)')
ax2.set_title('PCA Projection: Red vs White Wine')
ax2.legend()

plt.tight_layout()
plt.show()

## 7. K-Means on PCA-Reduced Features

Same algorithm (`kmeans2`), same seed — now applied to the 2-component PCA projection.

In [ ]:
pca_labels = run_kmeans(projected, k=N_CLUSTERS)
pca_acc = cluster_accuracy(y, pca_labels)
print(f'PCA accuracy (z-score + PCA): {pca_acc:.1%}')

## 8. Same Algorithm, Very Different Result

The preprocessing pipeline determines the outcome — not the algorithm.

To reproduce the *better* result you need to version everything: the pipeline code, the random seed, the data, and the environment. That is what the rest of this workshop is about.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

bar_labels = ['Raw features\n(no preprocessing)', 'Z-score + PCA\n(standardised)']
accuracies = [raw_acc * 100, pca_acc * 100]
bar_colors = ['#e74c3c', '#2ecc71']

bars = ax.bar(bar_labels, accuracies, color=bar_colors, width=0.45)
ax.set_ylim(0, 115)
ax.set_ylabel('Cluster Accuracy (%)')
ax.set_title('K-Means Accuracy: Raw vs Preprocessed', fontsize=13)
ax.axhline(50, color='gray', linestyle='--', alpha=0.5, label='Random guess (50%)')

for bar, acc in zip(bars, accuracies):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 2,
        f'{acc:.1f}%',
        ha='center',
        fontsize=13,
        fontweight='bold',
    )

ax.legend()
plt.tight_layout()
plt.show()

print(f'\nSame algorithm. Same data.')
print(f'Raw accuracy:  {raw_acc:.1%}')
print(f'PCA accuracy:  {pca_acc:.1%}')
print(f'Difference:    +{pca_acc - raw_acc:.1%}')
print('\nTo reproduce either result you need to version the full pipeline — not just the code.')